# Lecture 13 - Objects and Polymorphism (https://bit.ly/intro_python_13)

* Python objects:
  * Polymorphism: one operation, many types
  * How Python decides what `+`, `len()` and `print()` do ("dunder" methods)
  * Making objects printable: `__str__` and `__repr__`
  * How Python implements `==` for objects
  * General operator overloading, including comparison operators (with playing cards)
  * Copying objects, and a bug that shallow copies can cause

# Warm-up: predict the output

Before running the next cell, write down what you think each line will print.

In [ ]:
print(3 * 3)
print("ha" * 3)
print([0] * 3)

print(len("GATTACA"))
print(len([10, 20, 30]))
print(len({"A": 1, "C": 2}))

The same operator (`*`) or function (`len()`) did something different depending on the type of thing it was given. That idea has a name: polymorphism.

# Polymorphism

* "the condition of occurring in several different forms."

* In programming, polymorphism means the same operation behaves differently depending on the type of the object it is applied to:

<img src="https://raw.githubusercontent.com/benedictpaten/intro_python/main/lecture_notebooks/figures/graffles/polymorphism.jpg" width=700 height=350 />

* Polymorphism is everywhere in Python, and is a big part of why Python is pleasant to use. The `+` operator is another example:

In [2]:
# Okay, so getting back to code..

print(10 + 5)              # adds numbers
print("hello " + "world")  # joins strings
print([1, 2] + [3])        # joins lists

15
hello world
[1, 2, 3]


# So how does Python know what to do?

When Python sees `x + y`, it looks up a method called `__add__` on the type of `x` and calls it. So `x + y` really means `x.__add__(y)`.

In the same way, `len(x)` calls `x.__len__()`, and `str(x)` calls `x.__str__()`.

Methods with two underscores on each side are often called "dunder" (double underscore) methods. Each type supplies its own version, which is how the same operator can do different things:

In [3]:
print((10).__add__(5))            # what 10 + 5 does
print("hello ".__add__("world"))  # what "hello " + "world" does
print([1, 2].__add__([3]))        # what [1, 2] + [3] does

15
hello world
[1, 2, 3]


This is the useful part for us: if we write these methods in our own classes, Python's operators and built-in functions will work on our objects too. The rest of the lecture is (mostly) about doing exactly that.

# Redefining the behavior of str()

Let's go back to the Point class we created last time.

Suppose we want a method that creates a printable string representing a point:

In [4]:
class Point:
  """ Create a new Point, at coordinates x, y """

  def __init__(self, x=0, y=0):
    """ Create a new point at x, y """
    self.x = x
    self.y = y

  def distance_from_origin(self):
    """ Compute my distance from the origin """
    return ((self.x ** 2) + (self.y ** 2)) ** 0.5  # Pythagoras's theorem

  def to_string(self):
    """Return a string representing the point"""
    return f"({self.x}, {self.y})"

In [5]:
p = Point(3, 4)

print(p.to_string())

(3, 4)


`to_string()` works, but what happens if we just call print on p?

In [6]:
print(p)  # Not very helpful

`print` calls `str()` on p to turn it into text, and the result is the same:

In [7]:
str(p)

'<__main__.Point object at 0x7fa2ab3e9be0>'

Every class automatically gets a default `__str__` method (next lecture, on inheritance, we'll see where it comes from). The default just reports the type and the memory address.

We can replace it with our own by defining `__str__`:

In [8]:
class Point:
  """ Create a new Point, at coordinates x, y """

  def __init__(self, x=0, y=0):
    """ Create a new point at x, y """
    self.x = x
    self.y = y

  def distance_from_origin(self):
    """ Compute my distance from the origin """
    return ((self.x ** 2) + (self.y ** 2)) ** 0.5

  def __str__(self):  # str() and print() call this
    """Return a string representing the point"""
    return f"({self.x}, {self.y})"

In [9]:
p = Point(3, 4)

str(p)

'(3, 4)'

In [10]:
print(p)  # Much better

(3, 4)


## A close relative: `__repr__`

Python actually has two ways to turn an object into text:

* `str(p)`, used by `print`, is meant to be readable by people.
* `repr(p)` is meant to be unambiguous, for programmers. Jupyter uses it when a cell ends with a bare expression, and lists use it to display the items they contain.

We only defined `__str__`, so watch what happens here:

In [11]:
p = Point(3, 4)
q = Point(1, 2)

print(p)       # uses __str__, so this is fine
print([p, q])  # the list shows its items using repr, so we're back to the ugly version

(3, 4)
[<__main__.Point object at 0x7fa2ab5e3390>, <__main__.Point object at 0x7fa2ab5e39d0>]


In [12]:
class Point:
  """ Create a new Point, at coordinates x, y """

  def __init__(self, x=0, y=0):
    self.x = x
    self.y = y

  def __str__(self):
    """Return a readable string representing the point"""
    return f"({self.x}, {self.y})"

  def __repr__(self):
    """Return a string that looks like the code to create the point"""
    return f"Point({self.x}, {self.y})"

[Point(3, 4), Point(1, 2)]

[Point(3, 4), Point(1, 2)]

A good habit is to give every class you write a `__repr__`. If a class has `__repr__` but no `__str__`, then `str()` and `print()` will use `__repr__` too.

# Challenge 1

In [6]:
class Food:
  def __init__(self, name, flavor):
    self.name = name
    self.flavor = flavor

  # Add a method to the above class definition so that the code below works 

f = Food("vindaloo", "spicy")
print("I am going to eat:", f)  # Should print "I am going to eat: vindaloo which is spicy"

I am going to eat: vindaloo which is spicy


# How Python implements "==" for objects

As another example of polymorphism, let's revisit `==` and the `is` keyword and see how Python implements `==`.

Recall from earlier that:

* `==` tests whether two things are equivalent
* `is` tests whether two references point to the same object in memory

In [14]:
x = [1, 2]
y = [1, 2]

x == y  # Python checks that they represent the same thing

True

In [15]:
x is y  # Python checks whether x and y refer to the same object

False

In [16]:
z = x  # Make another reference to the object x refers to

z is x

True

You can't change what `is` means: two references either point to the same object or they don't.

`==`, on the other hand, calls the `__eq__()` method, which we can define. First, let's see what happens without it:

In [17]:
class Point:
  """ Create a new Point, at coordinates x, y """

  def __init__(self, x=0, y=0):
    self.x = x
    self.y = y

  def __repr__(self):
    return f"Point({self.x}, {self.y})"

p = Point(5, 10)
q = Point(5, 10)

In [18]:
p == q  # False! By default, __eq__() on objects behaves like 'is'

False

In [19]:
p == p  # True, because p is p

True

In [20]:
# Now let's implement __eq__

class Point:
  """ Create a new Point, at coordinates x, y """

  def __init__(self, x=0, y=0):
    self.x = x
    self.y = y

  def __repr__(self):
    return f"Point({self.x}, {self.y})"

  def __eq__(self, other):
    """ True if the points have the same x, y coordinates """
    return (self.x, self.y) == (other.x, other.y)

p = Point(5, 10)
q = Point(5, 10)

print(p is q)  # p is not q: they are distinct objects in memory
print(p == q)  # but they are now equal

False
True


In [21]:
# We get != (not equals) for free: Python uses the opposite of __eq__

p != q

False

What happens if we compare a Point with something that isn't a Point?

In [22]:
p == (5, 10)

AttributeError: 'tuple' object has no attribute 'x'

Our `__eq__` assumed `other` has `.x` and `.y` attributes, and a tuple doesn't.

A more careful `__eq__` checks the type first. `isinstance(obj, Point)` is True if `obj` is a Point. If `other` isn't something we know how to compare, we return the special value `NotImplemented`. That tells Python "I don't know how to do this"; Python then tries asking the other object, and if that doesn't work either, it falls back to `is`, which gives False.

In [23]:
class Point:
  """ Create a new Point, at coordinates x, y """

  def __init__(self, x=0, y=0):
    self.x = x
    self.y = y

  def __repr__(self):
    return f"Point({self.x}, {self.y})"

  def __eq__(self, other):
    """ True if other is a Point with the same x, y coordinates """
    if not isinstance(other, Point):
      return NotImplemented
    return (self.x, self.y) == (other.x, other.y)

p = Point(5, 10)

print(p == Point(5, 10))
print(p == (5, 10))   # no crash now, just False
print(p == "hello")

True
False
False


# Challenge 2

In [9]:
# Expand the class definition you wrote in Challenge 1 to include an __eq__ method.
# The method should return True if the foods have the same name, otherwise it should return False
# It should handle the case that 

class Food:
  def __init__(self, name, flavor):
    self.name = name
    self.flavor = flavor
    
    # code to write
    


curry = Food("vindaloo", "spicy")
another_curry = Food("vindaloo", "hot")
taco = Food("fajita tacos", "zingy")

# The following should work
assert curry == another_curry  # equal because both vindaloo
assert curry != taco
assert another_curry != taco
assert curry != "vindaloo" # should work for comparison to a string
assert curry != None


# General operator overloading

We've now seen two examples of polymorphism: we changed what `str()` and `==` do for our class by defining `__str__()` and `__eq__()`.

Defining `__eq__()` is an example of *operator overloading*: giving an existing operator a meaning for a new type.

We can do the same for the arithmetic operators. For points, adding two points together (like adding vectors) and scaling a point by a number both make sense:

In [25]:
class Point:
  """ Create a new Point, at coordinates x, y """

  def __init__(self, x=0, y=0):
    self.x = x
    self.y = y

  def __repr__(self):
    return f"Point({self.x}, {self.y})"

  def __add__(self, other):
    """ p + q: add two points coordinate by coordinate """
    return Point(self.x + other.x, self.y + other.y)

  def __mul__(self, k):
    """ p * k: scale a point by a number k """
    return Point(self.x * k, self.y * k)

p = Point(5, 10)
q = Point(2, 3)

print("p + q =", p + q)  # Point(5 + 2, 10 + 3)
print("p * 3 =", p * 3)  # Point(5 * 3, 10 * 3)

p + q = Point(7, 13)
p * 3 = Point(15, 30)


**Puzzle:** `p * 3` works. What do you think `3 * p` will do?

In [26]:
3 * p

TypeError: unsupported operand type(s) for *: 'int' and 'Point'

Python first tries `(3).__mul__(p)`. Integers don't know anything about Points, so that returns `NotImplemented`. Python then tries a "reflected" method on the right-hand object, `p.__rmul__(3)`, which we haven't written. Adding it takes an additional method:

In [27]:
class Point:
  """ Create a new Point, at coordinates x, y """

  def __init__(self, x=0, y=0):
    self.x = x
    self.y = y

  def __repr__(self):
    return f"Point({self.x}, {self.y})"

  def __add__(self, other):
    return Point(self.x + other.x, self.y + other.y)

  def __mul__(self, k):
    return Point(self.x * k, self.y * k)

  def __rmul__(self, k):
    """ k * p: called when the number is on the left """
    return self * k

p = Point(5, 10)
print("3 * p =", 3 * p)

3 * p = Point(15, 30)


# Operator overloading in the wild

Libraries lean on this all the time. One nice example is the `pathlib` module, which uses `/` to build file paths that work on any operating system:

In [28]:
from pathlib import Path

data_dir = Path("data")
reads = data_dir / "sample1" / "reads.fastq"  # this calls Path.__truediv__

print(reads)
print(reads.name, reads.suffix)

data/sample1/reads.fastq
reads.fastq .fastq


Similarly, time/date stamps:

In [1]:
from datetime import date, timedelta

print(date(2026, 11, 9) + timedelta(days=30))  # date arithmetic, via __add__
print({1, 2} | {2, 3})                         # set union, via __or__
print({1, 2} & {2, 3})  # set intersection

2026-12-09
{1, 2, 3}
{2}


numpy, which many of you will use later, does the same thing so that `a + b` adds two whole arrays element by element.

# Cheat sheet: what each method unlocks

| If you define... | ...Python lets you write |
|---|---|
| `__str__` | `str(x)`, `print(x)`, f-strings |
| `__repr__` | `repr(x)`; how `x` appears in Jupyter and inside lists |
| `__eq__` | `x == y` (and `x != y` for free) |
| `__lt__` | `x < y`, and also `sorted()`, `min()`, `max()` |
| `__le__`, `__lt__`, `__gt__`, `__ge__` | `<=`, `<`, `>`, `>=` |
| `__add__`, `__sub__`, `__mul__`, `__truediv__`, `__floordiv__` | `+`, `-`, `*`, `/`, `//` |
| `__len__` | `len(x)` |
| `__contains__` | `item in x` |
| `__getitem__` | `x[i]` |

This is definitely not exhaustive, the full list is in the Python docs: https://docs.python.org/3/reference/datamodel.html#special-method-names

# Challenge 3

In [ ]:
# Complete the code to make the asserts pass

class Contact:
  """Used to represent the contact details of an individual"""

  def __init__(self, name, phone, email):
    self.name, self.phone, self.email = name, phone, email

  # Implement equals, less-than, less-than-or-equals, greater-than and greater-than-or-equals
  # comparison methods, comparing only by the name attribute.
  # (You can write each one by hand - note there are clever ways to do this with the @total_ordering decorator - but
  # beyond the scope here.)

f = Contact("Dave", 1234, "dave@dave.net")
g = Contact("Dave", 56, "an_email@email")
h = Contact("Ben", 439205934, "ggg")


assert f == g
assert f != h
assert h < f and h < g and h <= f and h <= g  # Because Ben comes before Dave in the dictionary
assert f > h and g > h and f >= h and g >= h  # Because Ben comes before Dave in the dictionary



# Copying Objects

* We've talked about how different objects of a class are distinct. 

* What if we want to copy an object?

  * If the object is immutable you never need copy it, because you can just make multiple references to it without fear of any reference altering it.

  * With mutable objects it is sometimes useful to actually copy the object, effectively cloning the contents of the memory.

We'll illustrate how this works:

In [11]:
import copy

p1 = Point(3, 4)
p2 = copy.copy(p1)

p1 == p2

True

In [ ]:
p1 is p2 # Nope, p2 is a copy of p1 so the memory references are different

False

* Q: How does this work?

  * Python copies the contents of the p1 object's memory to a new location in memory, making copies of its references
  * e.g. p1 is composed of memory representing p1.x and p1.y
  * p2 is therefore a new location in memory with copies of these references. i.e. p1.x is p2.x and p1.y is p2.y
  

In [12]:
p1.x is p2.x # This is true because the references of p1 are copied into p2

True

In [13]:
p1.y is p2.y # Similarly

True

In [14]:
# It follows that if we reassign the variables in p1, this does not affect the references p2

p1.x = 7
p1.y = 12


print("p1", p1)
print("p2", p2) 

p1 (7, 12)
p2 (3, 4)


In [15]:
p1.x is p2.x # Having changed p1.x, p1.x is no longer p2.x because p1.x is pointing at 7 not 3

False

In [16]:
p1.y is p2.y # Similarly

False

# Copying Recursively

* Ok, but what about when we want to copy the references, recursively? 

<img src="https://raw.githubusercontent.com/benedictpaten/intro_python/main/lecture_notebooks/figures/graffles/deep%20copy.jpg" width=600 height=300 />

* If we use copy we only clone the original objects, not the referenced objects, so the reference object is shared.

* Consider the following example:

In [39]:
class Character:
  """ A game character with a name and an inventory of items """

  def __init__(self, name, inventory):
    self.name = name
    self.inventory = inventory

  def __repr__(self):
    return f"{self.name} carrying {self.inventory}"

hero = Character("Ada", ["torch", "map"])

clone = copy.copy(hero)
clone.name = "Ada's clone"
clone.inventory.append("sword")  # give the clone a sword

print(hero)
print(clone) # shoot, note the clone has the same inventory as Ada

Ada carrying ['torch', 'map', 'sword']
Ada's clone carrying ['torch', 'map', 'sword']


The original hero has a sword too!

Changing `clone.name` was fine, because it pointed `clone.name` at a new string. But `append` changed the list itself, and the shallow copy left both characters sharing one list:

In [40]:
hero.inventory is clone.inventory

True

It helps a lot to see this as a picture. (Note to self: Draw a picture!)

# Deep copy

`copy.deepcopy()` copies the object, then copies each object it refers to, then each object *those* refer to, and so on all the way down. It also keeps track of what it has already copied, so an object that is referred to twice is only copied once.

In [41]:
hero = Character("Ada", ["torch", "map"])

clone = copy.deepcopy(hero) # copy.deepcopy() copies the object, then copies each object it refers to, 
# then each object those refer to, and so on all the way down. It also keeps track of what 
# it has already copied, so an object that is referred to twice is only copied once.
clone.name = "Ada's clone"
clone.inventory.append("sword")

print(hero)
print(clone)
print(hero.inventory is clone.inventory)

Ada carrying ['torch', 'map']
Ada's clone carrying ['torch', 'map', 'sword']
False


Deep copies take more time and memory than shallow ones, so use `deepcopy` when you need a fully independent copy of an object that contains mutable things, and `copy` otherwise.

# Challenge 4

In [ ]:
class Contact:
  """Used to represent the contact details of an individual"""

  def __init__(self, name, phone, email):
    self.name, self.phone, self.email = name, phone, email

f = Contact("Dave", 1234, "dave@dave.net")


# Use the copy module to make a copy, g, of f and replace the email address of g to be "an_email@email"
# Check that the two copies have distinct email addresses.

# Reading

* Finish reading Chapter 16 (more objects): http://openbookproject.net/thinkcs/python/english3e/classes_and_objects_II.html
* Read Chapter 21 (polymorphism): http://openbookproject.net/thinkcs/python/english3e/even_more_oop.html

# Homework

* Go to Canvas and complete the lecture quiz, which involves completing each challenge problem
* Zybooks Reading 13
* Study ahead: work through the next lecture's notes on your own: An Example of OOP: Implementing the card game Old Maid. This will make it easier to follow.

# Practice Python

In [ ]:
# Problem 1: String Representation
# Complete the Employee class by implementing the __str__ method to return a string in the format:
# "Employee: [name] - [title]"

class Employee:
  def __init__(self, name, title):
    self.name = name
    self.title = title

  # Add your __str__ method here

# Tests
e = Employee("John Smith", "Software Engineer")
assert str(e) == "Employee: John Smith - Software Engineer"

In [ ]:
# Problem 2: Operator Overloading
# Complete the Temperature class by implementing the __eq__ and __lt__ methods
# Two temperatures should be equal if their Celsius values are equal
# One temperature is less than another if its Celsius value is less

class Temperature:
  def __init__(self, celsius):
    self.celsius = celsius

  # Add your __eq__ method here

  # Add your __lt__ method here

# Tests
t1 = Temperature(20)
t2 = Temperature(20)
t3 = Temperature(25)
assert t1 == t2
assert t1 < t3

In [ ]:
# Problem 3: Container Class
# Complete the Playlist class that contains songs. Implement __len__ to return the number
# of songs and __contains__ to check if a song is in the playlist.

class Playlist:
  def __init__(self, name):
    self.name = name
    self.songs = []

  def add_song(self, song):
    self.songs.append(song)

  # Add your __len__ method here

  # Add your __contains__ method here

# Tests
playlist = Playlist("My Favorites")
playlist.add_song("Bohemian Rhapsody")
playlist.add_song("Stairway to Heaven")
assert len(playlist) == 2
assert "Bohemian Rhapsody" in playlist
assert "Sweet Child O Mine" not in playlist

In [ ]:
# Problem 4: Time Arithmetic
# Complete the Time class by implementing the __add__ method to add minutes to a time.
# The time should wrap around correctly (e.g., 11:50 + 20 minutes = 12:10)

class Time:
  def __init__(self, hours, minutes):
    self.hours = hours
    self.minutes = minutes

  def __str__(self):
    return f"{self.hours:02d}:{self.minutes:02d}"

  # Add your __add__ method here

# Tests
t = Time(11, 50)
t2 = t + 20  # Should add 20 minutes
assert str(t2) == "12:10"
t3 = Time(23, 45) + 30
assert str(t3) == "00:15"

In [ ]:
# Problem 5: Complex Number Operations
# Complete the ComplexNum class by implementing __add__ and __mul__ methods
# for addition and multiplication of complex numbers

class ComplexNum:
  def __init__(self, real, imag):
    self.real = real
    self.imag = imag

  def __str__(self):
    return f"{self.real} + {self.imag}i"

  # Add your __add__ method here

  # Add your __mul__ method here

# Tests
c1 = ComplexNum(1, 2)
c2 = ComplexNum(3, 4)
sum_c = c1 + c2
assert sum_c.real == 4 and sum_c.imag == 6
prod_c = c1 * c2
assert prod_c.real == -5 and prod_c.imag == 10  # (1+2i)(3+4i) = (1*3-2*4) + (1*4+2*3)i = -5 + 10i

In [ ]:
# Problem 6: DNA Sequences
# Complete the Sequence class so that the tests pass. You will need:
#   len(s)       -> the number of bases
#   s1 + s2      -> a new Sequence with the bases joined (name it however you like)
#   "TTA" in s   -> True if the motif appears in the sequence
#   str(s)       -> FASTA style: ">name" on one line, then the bases on the next
#   s1 < s2      -> compare by GC content (use the gc_content method)

class Sequence:
  def __init__(self, name, bases):
    self.name = name
    self.bases = bases

  def gc_content(self):
    """ Fraction of bases that are G or C """
    return (self.bases.count("G") + self.bases.count("C")) / len(self.bases)

  # Add your methods here

# Tests
s1 = Sequence("seq1", "GATTACA")
s2 = Sequence("seq2", "GGCC")
assert len(s1) == 7
assert "TTA" in s1 and "CCC" not in s1
joined = s1 + s2
assert joined.bases == "GATTACAGGCC" and len(joined) == 11
assert str(s2) == ">seq2\nGGCC"
assert s1 < s2  # GATTACA is 2/7 G or C, GGCC is 4/4